In [253]:
import pandas as pd
import os
import glob
import re

**CLEAN THE 18 REMAINING excel file format**

In [8]:
xlsx_files = glob.glob("data_18 excel file/*.xls")
print(len(xlsx_files))
print(xlsx_files[:5])

0
[]


In [10]:
# Load the QTAX file - "qtax_2015_q1.xls"
qtax = pd.read_excel("data_18 excel file/qtax_2022_q1.xlsx")

In [20]:
df = pd.read_excel(
    "data_18 excel file/qtax_2022_q1.xlsx",
    header=5
)

print(df.head())
print(df.columns)

                                 Tax Description Code Unnamed: 2  \
0                                    Total Taxes  NaN         r/   
1                                 Property taxes  T01        NaN   
2                 Sales and gross receipts taxes  NaN        NaN   
3               General sales and gross receipts  T09         r/   
4       Selective sales and gross receipts taxes  NaN        NaN   

   U.S. Total (excludes Washington, D.C.)   0 Alabama*  0.1 Alaska*  0.2  \
0                             366755002.0 NaN  3864906  NaN  687188   r/   
1                               5414667.0 NaN   211107  NaN      19  NaN   
2                                     NaN NaN      NaN  NaN     NaN  NaN   
3                             102837531.0 NaN   731653  NaN       X  NaN   
4                                     NaN NaN      NaN  NaN     NaN  NaN   

  Arizona*  ... 0.46  Washington 0.47 West Virginia*  0.48 Wisconsin 0.49  \
0  5116262  ...   r/     9416448   r/      1746171.0   Na

In [22]:
df.columns = (
    df.columns
    .str.replace("*", "", regex=False)
    .str.strip()
)

In [24]:
# Drop ALL numeric column names
df = df.loc[:, ~df.columns.astype(str).str.match(r"^\d+(\.\d+)?$")]

In [26]:
# Clean column list
print(df.columns.tolist())

['Tax Description', 'Code', 'Unnamed: 2', 'U.S. Total (excludes Washington, D.C.)', nan, 'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming', 'Washington, D.C.']


In [28]:
# Fix numeric values (optional but recommended)
cols = df.columns.drop(["Tax Description", "Code"])

df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

C:\Users\Hong Nhien Nguyen\AppData\Local\Temp\ipykernel_8704\2593552395.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")


In [30]:
# Reshape to long format (useful for dashboards)
df_long = df.melt(
    id_vars=["Tax Description", "Code"],
    var_name="State",
    value_name="Tax_Value"
)

print(df_long.head())

                                 Tax Description Code       State  Tax_Value
0                                    Total Taxes  NaN  Unnamed: 2        NaN
1                                 Property taxes  T01  Unnamed: 2        NaN
2                 Sales and gross receipts taxes  NaN  Unnamed: 2        NaN
3               General sales and gross receipts  T09  Unnamed: 2        NaN
4       Selective sales and gross receipts taxes  NaN  Unnamed: 2        NaN


In [32]:
# remove U.S. Total column
# clean column names first
df.columns = df.columns.map(str)

# then filter safely
id_vars = ["Tax Description", "Code"]

state_cols = [
    col for col in df.columns
    if col not in id_vars and "U.S. Total" not in col
]

In [34]:
# Remove the non-state columns
# identify real state columns
id_vars = ["Tax Description", "Code"]

state_cols = [col for col in df.columns if col not in id_vars]
print(state_cols[:10])

# melt again to get the long table format

df_long = df.melt(
    id_vars=["Tax Description", "Code"],
    value_vars=state_cols,
    var_name="State",
    value_name="Tax_Value"
)

print(df_long.head())

['Unnamed: 2', 'U.S. Total (excludes Washington, D.C.)', 'nan', 'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut']
                                 Tax Description Code       State  Tax_Value
0                                    Total Taxes  NaN  Unnamed: 2        NaN
1                                 Property taxes  T01  Unnamed: 2        NaN
2                 Sales and gross receipts taxes  NaN  Unnamed: 2        NaN
3               General sales and gross receipts  T09  Unnamed: 2        NaN
4       Selective sales and gross receipts taxes  NaN  Unnamed: 2        NaN


In [36]:
# Create df_long
df_long = df.melt(
    id_vars=["Tax Description", "Code"],
    value_vars=state_cols,
    var_name="State",
    value_name="Tax_Value"
)

In [38]:
# Clean the tax descriptions
df_long["Tax Description"] = (
    df_long["Tax Description"]
    .str.replace("\xa0", "", regex=True)
    .str.strip()
)

In [40]:
# Define the tax categories you want
keep_taxes = [
    "Property taxes",
    "General sales and gross receipts",
    "Individual income",
    "Individual income taxes",
    "Corporation net income",
    "Corporate income taxes"
]

In [42]:
# Create df_long_clean
df_long_clean = df_long[
    df_long["Tax Description"].isin(keep_taxes)
].copy()

In [46]:
# Now create Tax_Group
df_long_clean["Tax_Group"] = df_long_clean["Tax Description"].replace({
    "Property taxes": "Property Tax",
    "General sales and gross receipts": "Sales Tax",
    "Individual income": "Individual Income Tax",
    "Individual income taxes": "Individual Income Tax",
    "Corporation net income": "Corporate Income Tax",
    "Corporate income taxes": "Corporate Income Tax"
})

In [48]:
# Add time info (Quater and year)
df_long["Year"] = 2022
df_long["Quarter"] = "Q1"

In [50]:
# Check if df_long_clean exists
print(type(df_long_clean))

<class 'pandas.core.frame.DataFrame'>


In [52]:
df_final = df_long_clean.pivot_table(
    index="State",
    columns="Tax_Group",
    values="Tax_Value",
    aggfunc="sum"
).reset_index()

In [54]:
df_final["total_tax"] = (
    df_final["Property Tax"] +
    df_final["Sales Tax"] +
    df_final["Individual Income Tax"] +
    df_final["Corporate Income Tax"]
)

In [56]:
print(df_final.columns.tolist())

['State', 'Corporate Income Tax', 'Individual Income Tax', 'Property Tax', 'Sales Tax', 'total_tax']


In [58]:
print(df_final.columns.tolist())
df_final.head(10)

['State', 'Corporate Income Tax', 'Individual Income Tax', 'Property Tax', 'Sales Tax', 'total_tax']


Tax_Group,State,Corporate Income Tax,Individual Income Tax,Property Tax,Sales Tax,total_tax
0,Alabama,105379.0,1775368.0,211107.0,731653.0,2823507.0
1,Alaska,18191.0,0.0,19.0,0.0,18210.0
2,Arizona,139702.0,1279438.0,260387.0,2661164.0,4340691.0
3,Arkansas,62960.0,864221.0,159326.0,1141310.0,2227817.0
4,California,14763814.0,42861282.0,782850.0,13021521.0,71429467.0
5,Colorado,151437.0,2407173.0,0.0,1048808.0,3607418.0
6,Connecticut,1279205.0,2563515.0,0.0,1548519.0,5391239.0
7,Delaware,91047.0,624452.0,0.0,0.0,715499.0
8,Florida,403870.0,0.0,0.0,9927935.0,10331805.0
9,Georgia,290783.0,4322813.0,187021.0,2042401.0,6843018.0


In [60]:
output_file = "data_clean2/qtax_2025_q4_clean.csv"  

df_final.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: data_clean2/qtax_2025_q4_clean.csv


**DEVELOP AN AUTOMATED PROCESS TO CLEAN THE REMAINING 17 EXCEL FILES**

In [64]:
import pandas as pd
import os
import glob
import re

# -----------------------------
# Folders
# -----------------------------
input_folder = "data_18 excel file"
output_folder = "data_clean2"
os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# Get all files
# -----------------------------
files = glob.glob(f"{input_folder}/*.xlsx")

print("Total files:", len(files))

# -----------------------------
# Loop through all files
# -----------------------------
for file in files:

    print(f"\nProcessing: {os.path.basename(file)}")

    # -----------------------------
    # Extract year & quarter from filename
    # -----------------------------
    match = re.search(r"qtax_(\d{4})_(q\d)", file, re.IGNORECASE)

    if match:
        year = int(match.group(1))
        quarter = match.group(2).upper()
    else:
        year = None
        quarter = None

    # -----------------------------
    # Read file
    # -----------------------------
    df = pd.read_excel(file, header=5)

    # Clean column names
    df.columns = df.columns.astype(str)
    df.columns = df.columns.str.replace("*", "", regex=False).str.strip()

    # Remove numeric columns (0, 0.1, etc.)
    df = df.loc[:, ~df.columns.str.match(r"^\d+(\.\d+)?$")]

    # Convert values
    cols = df.columns.drop(["Tax Description", "Code"])
    df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

    # -----------------------------
    # Identify state columns
    # -----------------------------
    id_vars = ["Tax Description", "Code"]

    state_cols = [
        col for col in df.columns
        if col not in id_vars and "U.S. Total" not in col
    ]

    # -----------------------------
    # Reshape to long
    # -----------------------------
    df_long = df.melt(
        id_vars=id_vars,
        value_vars=state_cols,
        var_name="State",
        value_name="Tax_Value"
    )

    # Clean text
    df_long["Tax Description"] = (
        df_long["Tax Description"]
        .str.replace("\xa0", "", regex=True)
        .str.strip()
    )

    # -----------------------------
    # Keep only needed taxes
    # -----------------------------
    keep_taxes = [
        "Property taxes",
        "General sales and gross receipts",
        "Individual income",
        "Individual income taxes",
        "Corporation net income",
        "Corporate income taxes"
    ]

    df_long_clean = df_long[
        df_long["Tax Description"].isin(keep_taxes)
    ].copy()

    # -----------------------------
    # Map categories
    # -----------------------------
    df_long_clean["Tax_Group"] = df_long_clean["Tax Description"].replace({
        "Property taxes": "Property Tax",
        "General sales and gross receipts": "Sales Tax",
        "Individual income": "Individual Income Tax",
        "Individual income taxes": "Individual Income Tax",
        "Corporation net income": "Corporate Income Tax",
        "Corporate income taxes": "Corporate Income Tax"
    })

    # -----------------------------
    # Pivot to final format
    # -----------------------------
    df_final = df_long_clean.pivot_table(
        index="State",
        columns="Tax_Group",
        values="Tax_Value",
        aggfunc="sum"
    ).reset_index()

    # Fill missing columns
    for col in ["Property Tax", "Sales Tax", "Individual Income Tax", "Corporate Income Tax"]:
        if col not in df_final.columns:
            df_final[col] = 0

    # Add time
    df_final["Year"] = year
    df_final["Quarter"] = quarter

    # Total tax
    df_final["Total Tax"] = (
        df_final["Property Tax"] +
        df_final["Sales Tax"] +
        df_final["Individual Income Tax"] +
        df_final["Corporate Income Tax"]
    )

    # Final order
    df_final = df_final[
        ["State", "Year", "Quarter",
         "Property Tax", "Sales Tax",
         "Individual Income Tax", "Corporate Income Tax",
         "Total Tax"]
    ]

    # -----------------------------
    # Save file
    # -----------------------------
    output_file = os.path.join(
        "data_clean2",
        os.path.basename(file).replace(".xlsx", "_clean.csv")
    )

    df_final.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")

print("\nALL FILES PROCESSED SUCCESSFULLY")

Total files: 18

Processing: qtax_2021_q3.xlsx
Saved: data_clean2\qtax_2021_q3_clean.csv

Processing: qtax_2021_q4.xlsx
Saved: data_clean2\qtax_2021_q4_clean.csv

Processing: qtax_2022_q1.xlsx
Saved: data_clean2\qtax_2022_q1_clean.csv

Processing: qtax_2022_q2.xlsx
Saved: data_clean2\qtax_2022_q2_clean.csv

Processing: qtax_2022_q3.xlsx
Saved: data_clean2\qtax_2022_q3_clean.csv

Processing: qtax_2022_q4.xlsx
Saved: data_clean2\qtax_2022_q4_clean.csv

Processing: qtax_2023_q1.xlsx
Saved: data_clean2\qtax_2023_q1_clean.csv

Processing: qtax_2023_q2.xlsx
Saved: data_clean2\qtax_2023_q2_clean.csv

Processing: qtax_2023_q3.xlsx
Saved: data_clean2\qtax_2023_q3_clean.csv

Processing: qtax_2023_q4.xlsx
Saved: data_clean2\qtax_2023_q4_clean.csv

Processing: qtax_2024_q1.xlsx
Saved: data_clean2\qtax_2024_q1_clean.csv

Processing: qtax_2024_q2.xlsx
Saved: data_clean2\qtax_2024_q2_clean.csv

Processing: qtax_2024_q3.xlsx
Saved: data_clean2\qtax_2024_q3_clean.csv

Processing: qtax_2024_q4.xlsx
Save